Remember to use `phoenix serve` to launch the server.

In [1]:
!uv pip install openinference-instrumentation-langchain opentelemetry-api opentelemetry-instrumentation
!uv pip install arize-phoenix-otel

PHOENIX_COLLECTOR_ENDPOINT = "http://localhost:4317" # grpc endpoint for the Phoenix collector

from phoenix.otel import register

# configure the Phoenix tracer
tracer_provider = register(
  project_name="default",
  auto_instrument=True,
  endpoint=PHOENIX_COLLECTOR_ENDPOINT,
)


Audited 3 packages in 2ms
Resolved 23 packages in 173ms                                        
Uninstalled 1 package in 3ms
Installed 1 package in 3ms                                  
 - protobuf==6.31.1
 + protobuf==5.29.5
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
  messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [ ]:
!uv pip install -qU "langchain[google-genai]"

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [ ]:
from typing import TypedDict, Literal

from langgraph.graph import StateGraph, END
from my_agent.utils.nodes import call_model, should_continue, tool_node, weather_guardrail
from my_agent.utils.state import AgentGuardrailBeforeState

# Define logic to determine whether question is about the weather
def is_about_weather(state: AgentGuardrailBeforeState) -> Literal['hardcoded_response', 'agent']:
    if not state['about_weather']:
        return "hardcoded_response"
    else:
        return "agent"

def hardcoded_response(state):
    return {"messages": [{"role": "assistant", "content": "sorry I can only answer questions about weather"}]}


# Define the config
class GraphConfig(TypedDict):
    model_name: Literal["anthropic", "openai"]


# Define a new graph
workflow = StateGraph(AgentGuardrailBeforeState, config_schema=GraphConfig)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)
workflow.add_node(weather_guardrail)
workflow.add_node(hardcoded_response)

# Set the entrypoint as `agent`
# This means that this node is the first one called
workflow.set_entry_point("weather_guardrail")

workflow.add_conditional_edges("weather_guardrail", is_about_weather)

# We now add a conditional edge
workflow.add_conditional_edges(
    # First, we define the start node. We use `agent`.
    # This means these are the edges taken after the `agent` node is called.
    "agent",
    # Next, we pass in the function that will determine which node is called next.
    should_continue,
    # Finally we pass in a mapping.
    # The keys are strings, and the values are other nodes.
    # END is a special node marking that the graph should finish.
    # What will happen is we will call `should_continue`, and then the output of that
    # will be matched against the keys in this mapping.
    # Based on which one it matches, that node will then be called.
    {
        # If `tools`, then we call the tool node.
        "continue": "action",
        # Otherwise we finish.
        "end": END,
    },
)

# We now add a normal edge from `tools` to `agent`.
# This means that after `tools` is called, `agent` node is called next.
workflow.add_edge("action", "agent")
workflow.add_edge("hardcoded_response", END)

# Finally, we compile it!
# This compiles it into a LangChain Runnable,
# meaning you can use it as you would any other runnable
graph = workflow.compile()